In [ ]:
!pip install -q transformers peft bitsandbytes accelerate pyngrok flask deep-translator

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from flask import Flask, request, jsonify
from pyngrok import ngrok
from deep_translator import GoogleTranslator

ngrok.set_auth_token("token")

HF_TOKEN = "token"
base_model_id = "google/gemma-2-2b-it"
adapter_id = "Edy317/my_restaurant_ai"

print("Incarcare AI...")
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=HF_TOKEN)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=quant_config, device_map="auto", token=HF_TOKEN)
model = PeftModel.from_pretrained(base_model, adapter_id, token=HF_TOKEN, subfolder="my_restaurant_ai")
print("Model incarcat.")

Incarcare AI...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Model incarcat.


In [14]:


app = Flask(__name__)
translator_to_en = GoogleTranslator(source='ro', target='en')
translator_to_ro = GoogleTranslator(source='en', target='ro')

@app.route('/generate', methods=['POST'])
def generate():
    data = request.json
    nume = data.get('nume', '')
    specific_ro = data.get('specific', '')

    try:
        specific_en = translator_to_en.translate(specific_ro) if specific_ro else ''

        prompt = f"Instruction: Write a short restaurant description.\nInput: Name: {nume}, Cuisine: {specific_en}\nOutput:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

        raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        final_description_en = raw_output.split("Output:")[-1].strip()

        final_description_ro = translator_to_ro.translate(final_description_en)

        return jsonify({'descriere': final_description_ro})

    except Exception as e:
        print(f"Eroare: {e}")
        fallback = f"Descoperă {nume}, noul tău loc preferat situat pe {specific_ro}. Te așteptăm cu preparate delicioase într-o atmosferă de neuitat!"
        return jsonify({'descriere': fallback})

public_url = ngrok.connect(5000).public_url
print(f"\nLink Django: {public_url}/generate\n")

app.run(port=5000)


Link Django: https://affiliate-rockiness-jaywalker.ngrok-free.dev/generate

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/May/2026 10:04:26] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 10:05:41] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 10:06:10] "POST /generate HTTP/1.1" 200 -
